# Sistema Knowledge Graph RAG — Trabajo Práctico 2
**Ingeniería Ontológica – 3010090 · Universidad Nacional de Colombia, Medellín**  
Profesor: Jaime Alberto Guzmán Luna

---

## Descripción
Sistema RAG agéntico avanzado sobre un corpus de **50 artículos científicos de arXiv** 
en el dominio de **Bioinformática y Medicina**, extendido con un **Knowledge Graph** 
construido sobre una ontología OWL biomédica.

## Tabla de contenidos
| Sección | Componente | Archivo individual |
|---|---|---|
| [0 · Setup](#setup) | Dependencias, Drive, API keys, LangSmith | — |
| [1 · Corpus + Chunking Semántico](#corpus) | SemanticChunker, FAISS | P2_01 |
| [2 · Transformación de Consultas](#transform) | HyDE, Query Decomposition | P2_02 |
| [3 · Recuperación Avanzada](#retrieval) | MMR, k dinámico | P2_03 |
| [4 · Agente ReAct + Reflecting](#agent) | LangGraph, 5 Tools, loop reflexivo | P2_04 |
| [5 · Knowledge Graph](#kg) | Ontología OWL, GraphDB, SPARQL, Inferencias | P2_05 |
| [6 · LangSmith + Métricas](#metrics) | Trazabilidad, Recall@k, MRR, Faithfulness | P2_06 |
| [7 · Casos de uso](#usecases) | 3 casos de uso completos | — |

## Arquitectura del sistema
```
    ┌──────────────────────────────────────────────────┐
    │                  USUARIO                         │
    └──────────────────┬───────────────────────────────┘
                       │ consulta
    ┌──────────────────▼───────────────────────────────┐
    │         TRANSFORMACIÓN DE CONSULTAS              │
    │  ┌─────────────┐    ┌──────────────────────────┐ │
    │  │ HyDE        │    │  Query Decomposition     │ │
    │  │ (< 10 palabras) │    │  (múltiples preguntas)   │ │
    │  └─────────────┘    └──────────────────────────┘ │
    └──────────────────┬───────────────────────────────┘
                       │
    ┌──────────────────▼───────────────────────────────┐
    │         AGENTE ReAct (LangGraph)                 │
    │  Decide autónomamente qué tools usar             │
    │  ┌────────────┬────────────┬────────────────────┐│
    │  │vector_search│ kg_query  │ decompose │ tavily ││
    │  │  (MMR FAISS)│ (SPARQL)  │           │ (web)  ││
    │  └────────────┴────────────┴────────────────────┘│
    └──────────────────┬───────────────────────────────┘
                       │ contexto
    ┌──────────────────▼───────────────────────────────┐
    │         GENERACIÓN (Gemini 2.0 Flash)            │
    └──────────────────┬───────────────────────────────┘
                       │ respuesta preliminar
    ┌──────────────────▼───────────────────────────────┐
    │    REFLEXIÓN/CRÍTICA (Gemini 2.0 Flash)          │
    │  score < 0.7 → retry (máx 3) → internet fallback│
    └──────────────────┬───────────────────────────────┘
                       │ métricas + trazabilidad LangSmith
    ┌──────────────────▼───────────────────────────────┐
    │              RESPUESTA FINAL                     │
    └──────────────────────────────────────────────────┘
```

## Asignación de LLMs
| Componente | LLM | Justificación |
|---|---|---|
| Agente ReAct (razonamiento) | **Gemini 2.0 Flash** | Razonamiento complejo multi-step; selección de tools con Tool Calling nativo |
| Transformación de consultas | **Gemini 2.0 Flash** | Comprensión semántica profunda para HyDE y detección de sub-preguntas |
| Generación de respuesta | **Gemini 2.0 Flash** | Síntesis de texto largo con contexto extenso; maneja ventanas de 1M tokens |
| Reflexión/Crítica | **Gemini 2.0 Flash** | Razonamiento crítico multi-criterio; detecta alucinaciones sutiles |
| k dinámico + Decomposición rápida | **Groq LLaMA 3.3 70B** | Latencia <500ms; ideal para micro-decisiones sin necesidad de razonamiento profundo |
| Búsqueda web (fallback) | Tavily API | Índice especializado en dominios académicos y científicos |

## 0 · Setup <a id='setup'></a>

In [ ]:
# ── Configuración local (reemplaza google.colab) ──────────────────────────
import sys
from pathlib import Path
# Ruta al repo resuelta desde este notebook (funciona para cualquier colaborador)
_REPO_ROOT = Path('__file__').parent.parent if '__file__' in dir() else Path().resolve()
# Buscar local_config.py subiendo directorios si hace falta
for _p in [_REPO_ROOT, _REPO_ROOT.parent, Path().resolve(), Path().resolve().parent]:
    if (_p / 'local_config.py').exists():
        _REPO_ROOT = _p
        break
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from local_config import (
    BASE_DIR, CORPUS_DIR, INDEX_DIR, TTL_PATH,
    GRAPHDB_BASE, REPO_NAME, GRAPHDB_URL,
    GOOGLE_API_KEY, GROQ_API_KEY, LANGCHAIN_API_KEY, TAVILY_API_KEY
)
print('✅ Configuración local cargada')
print(f'   BASE_DIR:    {BASE_DIR}')
print(f'   CORPUS_DIR:  {CORPUS_DIR}')
print(f'   GRAPHDB_URL: {GRAPHDB_URL}')


In [ ]:
import os, json, math, pickle, time, re, statistics, operator
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Annotated, TypedDict, Optional

# ── LangChain / LangGraph ─────────────────────────────────────────────────
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
from langchain.schema import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON, POST

# ── Modelos ───────────────────────────────────────────────────────────────
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/text-embedding-004',
    task_type='retrieval_document'
)
gemini = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.2, max_tokens=4096)
groq   = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0, max_tokens=512)

# ── GraphDB ───────────────────────────────────────────────────────────────
BIO = 'http://www.unal.edu.co/biomed#'

print('✅ Imports y modelos listos')
print(f'   GraphDB: {GRAPHDB_URL}')
print(f'   LangSmith project: {os.environ.get("LANGCHAIN_PROJECT", "?")}') 


## 1 · Corpus + Chunking Semántico <a id='corpus'></a>

Se reutiliza el corpus de 50 artículos de arXiv de la Práctica 1, pero ahora se divide
mediante **SemanticChunker** en lugar de tamaño fijo.

In [ ]:
import arxiv

def descargar_corpus(
    consulta: str = 'bioinformatics medicine deep learning genomics',
    n: int = 50
) -> None:
    """Descarga hasta `n` artículos de arXiv y guarda los PDFs en CORPUS_DIR."""
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)

    existing = list(CORPUS_DIR.glob('*.pdf'))
    if len(existing) >= n:
        print(f'📂 Corpus ya disponible: {len(existing)} PDFs en {CORPUS_DIR}')
        return

    print(f'⬇️  Descargando corpus ({n} artículos) → {CORPUS_DIR}')
    client = arxiv.Client(page_size=10, delay_seconds=3.0)
    search = arxiv.Search(
        query=consulta,
        max_results=n,
        sort_by=arxiv.SortCriterion.Relevance
    )

    descargados = 0
    for i, paper in enumerate(client.results(search)):
        arxiv_id = paper.entry_id.split('/')[-1].replace('/', '_')
        filename = f'doc_{i+1:02d}_{arxiv_id}.pdf'
        filepath = CORPUS_DIR / filename

        if filepath.exists():
            estado = 'ya existe'
        else:
            try:
                paper.download_pdf(dirpath=str(CORPUS_DIR), filename=filename)
                estado = 'descargado'
                descargados += 1
            except Exception as e:
                print(f'  [{i+1}] ⚠️  {paper.title[:60]} — {e}')
                continue

        print(f'  [{i+1:02d}/{n}] {estado:10s}  {paper.title[:65]}')

    total = len(list(CORPUS_DIR.glob('*.pdf')))
    print(f'\n✅ Corpus listo: {total} PDFs en {CORPUS_DIR}')


descargar_corpus()


In [ ]:
FAISS_PATH    = INDEX_DIR / 'faiss_semantic'
CHUNKS_PATH   = INDEX_DIR / 'semantic_chunks.pkl'

from langchain.text_splitter import RecursiveCharacterTextSplitter

def build_or_load_faiss_index():
    """Construye el índice FAISS con chunking semántico optimizado."""
    if FAISS_PATH.exists():
        print('📂 Cargando índice FAISS existente...')
        faiss_idx = FAISS.load_local(
            str(FAISS_PATH), embeddings,
            allow_dangerous_deserialization=True
        )
        with open(CHUNKS_PATH, 'rb') as f:
            all_chunks = pickle.load(f)
        print(f'✅ FAISS cargado: {faiss_idx.index.ntotal} vectores ({len(all_chunks)} chunks)')
        return faiss_idx, all_chunks
    
    print('🔨 Construyendo índice FAISS con SemanticChunker (2 pasos)...')
    
    # Paso 1: Dividir por tamaño fijo (rápido)
    splitter_coarse = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    
    # Paso 2: Aplicar SemanticChunker a trozos pequeños (mucho más rápido)
    semantic_chunker = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type='percentile',
        breakpoint_threshold_amount=95
    )
    
    pdfs = list(CORPUS_DIR.glob('*.pdf'))
    print(f'   Documentos: {len(pdfs)}')
    
    all_chunks = []
    for i, pdf in enumerate(pdfs):
        try:
            loader = PyPDFLoader(str(pdf))
            pages  = loader.load()
            text   = ' '.join([p.page_content for p in pages])
            
            # Paso 1: chunks toscos
            coarse = splitter_coarse.split_text(text)
            
            # Paso 2: semántico sobre cada trozo pequeño
            doc_id = pdf.stem
            chunk_idx = 0
            for coarse_chunk in coarse:
                try:
                    semantic_chunks = semantic_chunker.create_documents([coarse_chunk])
                    for c in semantic_chunks:
                        c.metadata = {
                            'doc_id': doc_id, 'source': str(pdf),
                            'chunk_id': f'{doc_id}_c{chunk_idx:03d}',
                            'chunk_idx': chunk_idx, 'char_count': len(c.page_content)
                        }
                        chunk_idx += 1
                        all_chunks.append(c)
                except Exception as e:
                    print(f'      ⚠️ Chunking semántico falló: {e} (guardando como está)')
                    doc = Document(page_content=coarse_chunk, metadata={
                        'doc_id': doc_id, 'source': str(pdf),
                        'chunk_id': f'{doc_id}_c{chunk_idx:03d}',
                        'chunk_idx': chunk_idx
                    })
                    chunk_idx += 1
                    all_chunks.append(doc)
            
            if (i+1) % 10 == 0:
                print(f'   [{i+1}/{len(pdfs)}] procesados ({len(all_chunks)} chunks total)')
        except Exception as e:
            print(f'   ⚠️ {pdf.name}: {e}')
    
    if not all_chunks:
        raise RuntimeError(
            f"No se encontraron documentos procesables en {CORPUS_DIR}. "
            "Ejecuta descargar_corpus() antes de construir el índice."
        )
    
    # Construir FAISS en lotes
    print(f'   Construyendo FAISS ({len(all_chunks)} chunks)...')
    faiss_idx = None
    for i in range(0, len(all_chunks), 50):
        batch = all_chunks[i:i+50]
        if faiss_idx is None:
            faiss_idx = FAISS.from_documents(batch, embeddings)
        else:
            faiss_idx.add_documents(batch)
    
    faiss_idx.save_local(str(FAISS_PATH))
    with open(CHUNKS_PATH, 'wb') as f:
        pickle.dump(all_chunks, f)
    
    print(f'✅ FAISS construido: {faiss_idx.index.ntotal} vectores ({len(all_chunks)} chunks)')
    return faiss_idx, all_chunks


faiss_index, all_chunks = build_or_load_faiss_index()


## 2 · Transformación de Consultas <a id='transform'></a>

In [ ]:
# ── HyDE ──────────────────────────────────────────────────────────────────
HYDE_PROMPT = ChatPromptTemplate.from_template("""
Escribe un párrafo técnico biomédico detallado (150-200 palabras) que sería la 
respuesta perfecta a esta consulta. Usa terminología científica precisa.
Consulta: {query}
Párrafo:
""")
hyde_chain = HYDE_PROMPT | gemini | StrOutputParser()


def apply_hyde(query: str, k: int = 5) -> List[Document]:
    """Genera un doc hipotético y lo usa para búsqueda semántica mejorada."""
    hypothetical_doc = hyde_chain.invoke({'query': query})
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': k, 'fetch_k': k*4, 'lambda_mult': 0.6}
    )
    return retriever.invoke(hypothetical_doc)


# ── Query Decomposition ───────────────────────────────────────────────────
DECOMP_PROMPT = ChatPromptTemplate.from_template("""
Descompone esta consulta compleja en 2-4 subconsultas simples separadas por '|'.
Solo devuelve las subconsultas, nada más.
Consulta: {query}
Subconsultas:
""")
decomp_chain = DECOMP_PROMPT | groq | StrOutputParser()


def decompose_query(query: str, k_per_sq: int = 3) -> List[Document]:
    """Descompone la consulta en subconsultas y recupera documentos para cada una."""
    sub_queries_str = decomp_chain.invoke({'query': query})
    sub_queries = [sq.strip() for sq in sub_queries_str.split('|') if sq.strip()]
    
    seen = set()
    all_docs = []
    for sq in sub_queries[:4]:
        docs = faiss_index.similarity_search(sq, k=k_per_sq)
        for d in docs:
            if d.metadata['chunk_id'] not in seen:
                seen.add(d.metadata['chunk_id'])
                all_docs.append(d)
    return all_docs


# ── Detector de tipo de consulta ──────────────────────────────────────────
class QueryType(BaseModel):
    needs_hyde: bool = Field(description="True si la consulta es corta/ambigua (< 10 palabras)")
    needs_decomposition: bool = Field(description="True si contiene múltiples preguntas")

detect_chain = (
    ChatPromptTemplate.from_template(
        'Analiza la consulta y determina la transformación necesaria.\nConsulta: "{query}"\n'
        'Reglas: needs_hyde=True si < 10 palabras O muy vaga; '
        'needs_decomposition=True si tiene AND/OR/además/también o múltiples preguntas.\n'
        'Solo JSON.'
    )
    | gemini.with_structured_output(QueryType)
)


def transform_and_retrieve(query: str, k: int = 5) -> tuple:
    """
    Router principal: analiza la consulta y aplica la transformación apropiada.
    Retorna (documentos, transform_type, metadata).
    """
    qtype = detect_chain.invoke({'query': query})
    
    if qtype.needs_hyde:
        docs = apply_hyde(query, k=k)
        return docs, 'hyde', {'word_count': len(query.split())}
    elif qtype.needs_decomposition:
        docs = decompose_query(query, k_per_sq=3)
        return docs, 'decomposition', {}
    else:
        retriever = faiss_index.as_retriever(
            search_type='mmr',
            search_kwargs={'k': k, 'fetch_k': k*4, 'lambda_mult': 0.6}
        )
        docs = retriever.invoke(query)
        return docs, 'direct', {}


print('✅ Transformación de consultas configurada')


## 3 · Tools del Agente <a id='retrieval'></a>

In [ ]:
def sparql_select(query: str) -> List[Dict]:
    """Ejecuta una consulta SPARQL SELECT en GraphDB."""
    try:
        sparql = SPARQLWrapper(GRAPHDB_URL)
        sparql.setQuery(query)
        sparql.setReturnFormat(SPARQL_JSON)
        result = sparql.query().convert()
        return [
            {k: v['value'].split('#')[-1] if '#' in v['value'] else v['value']
             for k, v in row.items()}
            for row in result['results']['bindings']
        ]
    except Exception as e:
        return [{'error': str(e)}]


# ── Tool 1: Búsqueda vectorial MMR ────────────────────────────────────────
@tool
def vector_search(query: str, k: int = 5) -> str:
    """
    Búsqueda semántica con MMR en el corpus biomédico (FAISS + SemanticChunker).
    Devuelve los fragmentos más relevantes y diversos.
    Úsala para recuperar información factual de artículos científicos.
    """
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': k, 'fetch_k': k*4, 'lambda_mult': 0.6}
    )
    docs = retriever.invoke(query)
    return '\n\n'.join([
        f'[{d.metadata["doc_id"]} | {d.metadata["chunk_id"]}]\n{d.page_content[:600]}'
        for d in docs
    ]) or 'No se encontraron resultados.'


# ── Tool 2: Knowledge Graph (SPARQL) ─────────────────────────────────────
@tool
def knowledge_graph_query(entity: str, relation_type: str = 'all') -> str:
    """
    Consulta la ontología biomédica en GraphDB para obtener relaciones estructuradas.
    Úsala para preguntas sobre relaciones entre genes, proteínas, enfermedades, fármacos.
    
    Args:
        entity: entidad a consultar (ej: 'BRCA1', 'BreastCancer', 'Trastuzumab')
        relation_type: 'treats', 'encodes', 'studiesDisease', o 'all'
    """
    q = f"""
    PREFIX bio: <{BIO}>
    SELECT ?s ?p ?o WHERE {{
        ?s ?p ?o .
        FILTER(
            CONTAINS(LCASE(STR(?s)), LCASE('{entity}')) ||
            CONTAINS(LCASE(STR(?o)), LCASE('{entity}'))
        )
        FILTER(!STRSTARTS(STR(?p), 'http://www.w3.org/1999/02/22-rdf-syntax-ns'))
    }} LIMIT 15
    """
    results = sparql_select(q)
    if not results or 'error' in results[0]:
        return f'No se encontraron relaciones para "{entity}" (verifique que GraphDB esté activo).'
    lines = [f'KG — Relaciones de "{entity}":']
    for r in results:
        lines.append(f'  {r.get("s","?")} --[{r.get("p","?")}]--> {r.get("o","?")}')
    return '\n'.join(lines)


# ── Tool 3: HyDE ──────────────────────────────────────────────────────────
@tool
def apply_hyde_tool(short_query: str) -> str:
    """
    Aplica HyDE a una consulta corta/ambigua: genera un documento hipotético
    y lo usa para recuperar fragmentos más relevantes del corpus.
    Úsala cuando la consulta tiene menos de 10 palabras o es muy vaga.
    """
    docs = apply_hyde(short_query, k=5)
    parts = [f'[HyDE: "{short_query}"]\']
    parts += [f'[{d.metadata["doc_id"]}]: {d.page_content[:400]}' for d in docs]
    return '\n\n'.join(parts)


# ── Tool 4: Decomposición + búsqueda ────────────────────────────────────
@tool
def decompose_and_search(complex_query: str) -> str:
    """
    Descompone una consulta compleja en subconsultas simples y busca cada una.
    Úsala cuando la consulta contiene múltiples preguntas o condiciones.
    """
    docs = decompose_query(complex_query, k_per_sq=3)
    return '\n\n'.join([
        f'[{d.metadata["doc_id"]}]: {d.page_content[:400]}'
        for d in docs
    ]) or 'Sin resultados.'


# ── Tool 5: Búsqueda web (fallback) ──────────────────────────────────────
tavily_search = TavilySearchResults(
    max_results=5,
    search_depth='advanced',
    include_domains=['pubmed.ncbi.nlm.nih.gov', 'nature.com', 'biorxiv.org'],
    description='Busca información biomédica actualizada en internet. SOLO usar como último recurso tras 3 intentos fallidos.'
)

TOOLS = [vector_search, knowledge_graph_query, apply_hyde_tool, decompose_and_search, tavily_search]

print('✅ 5 Tools definidas')
for t in TOOLS:
    print(f'   • {t.name}')


## 4 · Agente ReAct + Reflecting (LangGraph) <a id='agent'></a>

In [ ]:
# ── Estado ────────────────────────────────────────────────────────────────
class KGRAGState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    original_query: str
    retrieved_docs: List[str]
    current_answer: str
    reflection_score: float
    reflection_feedback: str
    attempt_count: int
    used_internet: bool
    trace: List[Dict[str, Any]]
    final_answer: str
    metrics: Dict[str, Any]


# ── LLM con tools ─────────────────────────────────────────────────────────
react_llm = gemini.bind_tools(TOOLS)

REACT_SYSTEM = """Eres un experto en bioinformática y medicina con acceso a:
1. Un corpus de 50 artículos científicos de arXiv (búsqueda vectorial MMR)
2. Una ontología biomédica en GraphDB (Knowledge Graph con genes, proteínas, enfermedades, fármacos)
3. Transformaciones de consulta: HyDE (preguntas vagas) y Decomposición (preguntas complejas)
4. Búsqueda web como último recurso

ESTRATEGIA:
- Para preguntas vagas/cortas: usa apply_hyde_tool
- Para preguntas complejas (AND/OR): usa decompose_and_search
- Para entidades biomédicas concretas: combina vector_search + knowledge_graph_query
- SIEMPRE cita las fuentes con [doc_id, chunk_id]
- tavily_search: SOLO si los otros métodos fallan 3 veces
"""


# ── Nodos ─────────────────────────────────────────────────────────────────
def react_agent_node(state: KGRAGState) -> KGRAGState:
    """Nodo ReAct: razona y decide qué herramientas invocar."""
    msgs = state['messages']
    if state['attempt_count'] > 0:
        msgs = msgs + [SystemMessage(
            content=f'[Reintento {state["attempt_count"]+1}/3] Feedback: {state["reflection_feedback"]}. '
                    'Usa estrategias diferentes para mejorar la respuesta.'
        )]
    response = react_llm.invoke([SystemMessage(content=REACT_SYSTEM)] + msgs)
    return {
        **state,
        'messages': [response],
        'trace': state.get('trace', []) + [{
            'node': 'react_agent', 'attempt': state['attempt_count']+1,
            'timestamp': datetime.now().isoformat(),
            'tool_calls': [tc['name'] for tc in (response.tool_calls or [])]
        }]
    }


tool_executor = ToolNode(TOOLS)

GEN_PROMPT = ChatPromptTemplate.from_messages([
    ('system', 'Eres un asistente biomédico. Responde SOLO con información del contexto proporcionado. '
               'Estructura la respuesta e incluye citas [doc_id].'),
    ('human', 'Pregunta: {query}\n\nContexto:\n{context}\n\nRespuesta detallada:')
])
gen_chain = GEN_PROMPT | gemini | StrOutputParser()


def generate_response_node(state: KGRAGState) -> KGRAGState:
    """Nodo de generación: construye la respuesta a partir del contexto."""
    context_parts = [
        msg.content for msg in state['messages']
        if hasattr(msg, 'content') and isinstance(msg.content, str) and len(msg.content) > 100
    ]
    context = '\n\n---\n\n'.join(context_parts[-6:])[:4000]
    
    answer = gen_chain.invoke({'query': state['original_query'], 'context': context})
    return {
        **state,
        'current_answer': answer,
        'retrieved_docs': context_parts,
        'trace': state.get('trace', []) + [{
            'node': 'generate', 'timestamp': datetime.now().isoformat(),
            'answer_chars': len(answer)
        }]
    }


REFLECT_PROMPT = ChatPromptTemplate.from_template("""
Evalúa esta respuesta biomédica:

Pregunta: {query}
Respuesta: {answer}
Contexto: {context}

Puntúa de 0 a 1:
- respaldada: ¿cada afirmación tiene respaldo en el contexto?
- coherente: ¿sin contradicciones?
- completa: ¿responde todos los aspectos de la pregunta?
- citaciones: ¿incluye referencias a las fuentes?

Devuelve SOLO JSON: {{"overall_score": 0.0, "feedback": "...", "approved": false}}
""")


def reflect_node(state: KGRAGState) -> KGRAGState:
    """Nodo reflexivo: evalúa la calidad de la respuesta generada."""
    reflect_chain = REFLECT_PROMPT | gemini | StrOutputParser()
    context = '\n'.join(state.get('retrieved_docs', [])[:3])[:2000]
    
    raw = reflect_chain.invoke({
        'query': state['original_query'],
        'answer': state['current_answer'],
        'context': context
    })
    
    try:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        ev = json.loads(m.group()) if m else {}
    except:
        ev = {}
    
    score    = float(ev.get('overall_score', 0.5))
    feedback = ev.get('feedback', raw[:300])
    
    print(f'  🔍 Reflexión #{state["attempt_count"]+1}: score={score:.2f} | {"✅" if score >= 0.7 else "🔄"}')
    
    return {
        **state,
        'reflection_score': score,
        'reflection_feedback': feedback,
        'attempt_count': state['attempt_count'] + 1,
        'trace': state.get('trace', []) + [{
            'node': 'reflect', 'score': score,
            'approved': score >= 0.7, 'feedback': feedback[:200],
            'timestamp': datetime.now().isoformat()
        }]
    }


def internet_fallback_node(state: KGRAGState) -> KGRAGState:
    """Fallback: busca en internet tras 3 intentos fallidos y retroalimenta la KB."""
    print('  🌐 Activando fallback web (3 intentos agotados)...')
    web_results = tavily_search.invoke(state['original_query'])
    web_ctx = '\n\n'.join([
        f'[WEB] {r.get("url","")}\n{r.get("content","")}'
        for r in (web_results if isinstance(web_results, list) else [{'content': str(web_results)}])
    ])
    answer = gen_chain.invoke({'query': state['original_query'], 'context': f'[Internet]\n{web_ctx}'})
    print(f'  📚 Conocimiento web indexado como: web_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
    return {
        **state, 'final_answer': answer, 'current_answer': answer, 'used_internet': True,
        'trace': state.get('trace', []) + [{
            'node': 'internet_fallback', 'timestamp': datetime.now().isoformat(),
            'web_results': len(web_results) if isinstance(web_results, list) else 1
        }]
    }


def finalize_node(state: KGRAGState) -> KGRAGState:
    """Nodo final: consolida la respuesta y calcula métricas básicas."""
    return {
        **state,
        'final_answer': state['current_answer'],
        'metrics': {
            'attempts': state['attempt_count'],
            'final_score': state['reflection_score'],
            'used_internet': state['used_internet'],
            'nodes_executed': [t['node'] for t in state.get('trace', [])]
        }
    }


# ── Enrutadores ───────────────────────────────────────────────────────────
def should_use_tools(state: KGRAGState) -> str:
    last = state['messages'][-1]
    return 'use_tools' if (hasattr(last, 'tool_calls') and last.tool_calls) else 'generate'

def after_reflect(state: KGRAGState) -> str:
    if state['reflection_score'] >= 0.7:
        return 'finalize'
    return 'internet_fallback' if state['attempt_count'] >= 3 else 'retry'


# ── Construir grafo ───────────────────────────────────────────────────────
g = StateGraph(KGRAGState)
g.add_node('react_agent',       react_agent_node)
g.add_node('tools',             tool_executor)
g.add_node('generate_response', generate_response_node)
g.add_node('reflect',           reflect_node)
g.add_node('internet_fallback', internet_fallback_node)
g.add_node('finalize',          finalize_node)

g.set_entry_point('react_agent')
g.add_conditional_edges('react_agent', should_use_tools, {'use_tools': 'tools', 'generate': 'generate_response'})
g.add_edge('tools', 'react_agent')
g.add_edge('generate_response', 'reflect')
g.add_conditional_edges('reflect', after_reflect, {'finalize': 'finalize', 'retry': 'react_agent', 'internet_fallback': 'internet_fallback'})
g.add_edge('internet_fallback', 'finalize')
g.add_edge('finalize', END)

kg_rag_graph = g.compile()

print('✅ Grafo KG-RAG compilado')
print('   Flujo: react → tools* → generate → reflect → [finalize | retry(×3) | web_fallback]')


In [ ]:
def run_kg_rag(query: str, verbose: bool = True) -> dict:
    """
    Función principal del sistema KG-RAG.
    
    Ejecuta el grafo completo: ReAct + Reflecting + KG + LangSmith.
    Retorna respuesta final con métricas y trazabilidad completa.
    """
    if verbose:
        print(f'\n{"="*65}')
        print(f'🤖 KG-RAG Query: "{query}"')
        print('='*65)
    
    state = kg_rag_graph.invoke({
        'messages': [HumanMessage(content=query)],
        'original_query': query,
        'retrieved_docs': [],
        'current_answer': '',
        'reflection_score': 0.0,
        'reflection_feedback': '',
        'attempt_count': 0,
        'used_internet': False,
        'trace': [],
        'final_answer': '',
        'metrics': {}
    })
    
    if verbose:
        print(f'\n📋 RESPUESTA:')
        print('-'*65)
        print(state['final_answer'])
        print('-'*65)
        m = state.get('metrics', {})
        print(f'  Intentos: {m.get("attempts","?")} | Score: {m.get("final_score", 0):.2f} | Web: {m.get("used_internet",False)}')
        print(f'  Nodos: {m.get("nodes_executed", [])}')
    
    return state


print('✅ Función run_kg_rag lista')


## 5 · Knowledge Graph <a id='kg'></a>

Setup de GraphDB y funciones de consulta SPARQL.

In [ ]:
import requests

def upload_ontology(ttl_path: Path, graphdb_url: str) -> bool:
    """Sube la ontología OWL (Turtle) a GraphDB."""
    if not ttl_path.exists():
        print(f'⚠️  TTL no encontrado: {ttl_path}')
        return False
    with open(ttl_path, 'rb') as f:
        resp = requests.post(
            f'{graphdb_url}/statements',
            data=f.read(),
            headers={'Content-Type': 'text/turtle; charset=UTF-8'}
        )
    ok = resp.status_code in (200, 204)
    print(f'{"✅" if ok else "❌"} Ontología en GraphDB: HTTP {resp.status_code}')
    return ok


TTL_PATH = BASE_DIR / 'biomedical_ontology.ttl'
if TTL_PATH.exists():
    upload_ontology(TTL_PATH, GRAPHDB_URL)
else:
    print('⚠️  Copia biomedical_ontology.ttl a /home/camilosanchez/Documentos/UN/ING Ontológica/RAG/')
    print('   El archivo está en: notebooks/biomedical_ontology.ttl')


# Test de conexión
try:
    n = sparql_select(f'SELECT (COUNT(*) AS ?c) WHERE {{ ?s ?p ?o }}')
    print(f'   Triples en GraphDB: {n[0]["c"] if n else "?"}')
except:
    print('   GraphDB no disponible — las tools de KG usarán modo degradado')


## 6 · LangSmith + Métricas <a id='metrics'></a>

In [ ]:
# ── Métricas de recuperación ──────────────────────────────────────────────
def precision_at_k(retrieved, relevant, k):
    return sum(1 for d in retrieved[:k] if d in set(relevant)) / k if k else 0.0

def recall_at_k(retrieved, relevant, k):
    return sum(1 for d in retrieved[:k] if d in set(relevant)) / len(relevant) if relevant else 0.0

def mrr(retrieved, relevant):
    for i, d in enumerate(retrieved, 1):
        if d in set(relevant): return 1.0/i
    return 0.0

def ndcg_at_k(retrieved, relevant, k):
    dcg  = sum(1/math.log2(i+1) for i, d in enumerate(retrieved[:k], 1) if d in set(relevant))
    idcg = sum(1/math.log2(i+1) for i in range(1, min(k, len(relevant))+1))
    return dcg/idcg if idcg else 0.0


# ── LLM como Juez ────────────────────────────────────────────────────────
class QualScore(BaseModel):
    relevance: float = Field(description="0-1: relevancia del contexto")
    faithfulness: float = Field(description="0-1: respuesta respaldada por el contexto")
    feedback: str = Field(description="Retroalimentación breve")

judge_chain = (
    ChatPromptTemplate.from_template(
        'Evalúa esta respuesta RAG biomédica:\n'
        'Pregunta: {query}\nContexto: {context}\nRespuesta: {answer}\n'
        'Puntúa relevance (0-1) y faithfulness (0-1). Solo JSON.'
    )
    | gemini.with_structured_output(QualScore)
)


def evaluate_response(query: str, context: str, answer: str) -> QualScore:
    """Evalúa la calidad de una respuesta RAG con Gemini como juez."""
    return judge_chain.invoke({
        'query': query, 'context': context[:1500], 'answer': answer
    })

print('✅ Métricas y LLM-juez configurados')
print(f'   LangSmith: https://smith.langchain.com → {os.environ["LANGCHAIN_PROJECT"]}')


## 7 · Casos de Uso <a id='usecases'></a>

Demostramos las funcionalidades del sistema con 3 casos de uso representativos.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CASO DE USO 1
# Tipo: Consulta directa sobre un gen y su enfermedad asociada
# Herramientas esperadas: vector_search + knowledge_graph_query
# LangSmith: registrará 2 tool calls + generación + reflexión
# ════════════════════════════════════════════════════════════════════════════

print('\n' + '█'*65)
print('CASO DE USO 1 — Consulta KG + Corpus')
print('█'*65)

result1 = run_kg_rag(
    'What is the role of BRCA1 gene in DNA repair and how does it '
    'relate to hereditary breast and ovarian cancer?'
)

# Evaluar calidad
context1 = '\n'.join(result1.get('retrieved_docs', [])[:3])[:2000]
if context1:
    score1 = evaluate_response(
        result1['original_query'], context1, result1['final_answer']
    )
    print(f'\n📊 Evaluación LLM-Juez:')
    print(f'   Relevance:    {score1.relevance:.2f}')
    print(f'   Faithfulness: {score1.faithfulness:.2f}')
    print(f'   Feedback: {score1.feedback[:120]}')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CASO DE USO 2
# Tipo: Consulta compleja multi-pregunta (activa Query Decomposition)
# Herramientas esperadas: decompose_and_search + knowledge_graph_query
# ════════════════════════════════════════════════════════════════════════════

print('\n' + '█'*65)
print('CASO DE USO 2 — Query Decomposition + KG')
print('█'*65)

result2 = run_kg_rag(
    'What genes are mutated in non-small cell lung cancer, '
    'what proteins do they encode, and what targeted therapies '
    'are currently approved for EGFR-positive NSCLC?'
)

context2 = '\n'.join(result2.get('retrieved_docs', [])[:3])[:2000]
if context2:
    score2 = evaluate_response(result2['original_query'], context2, result2['final_answer'])
    print(f'\n📊 Evaluación LLM-Juez:')
    print(f'   Relevance:    {score2.relevance:.2f}')
    print(f'   Faithfulness: {score2.faithfulness:.2f}')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CASO DE USO 3
# Tipo: Consulta corta/vaga (activa HyDE)
# Herramientas esperadas: apply_hyde_tool + vector_search
# ════════════════════════════════════════════════════════════════════════════

print('\n' + '█'*65)
print('CASO DE USO 3 — HyDE + Consulta corta')
print('█'*65)

result3 = run_kg_rag('Remdesivir COVID antiviral')

context3 = '\n'.join(result3.get('retrieved_docs', [])[:3])[:2000]
if context3:
    score3 = evaluate_response(result3['original_query'], context3, result3['final_answer'])
    print(f'\n📊 Evaluación LLM-Juez:')
    print(f'   Relevance:    {score3.relevance:.2f}')
    print(f'   Faithfulness: {score3.faithfulness:.2f}')


In [ ]:
# ── Resumen final del sistema ────────────────────────────────────────────
print('\n' + '='*65)
print('📊 RESUMEN DEL SISTEMA KG-RAG — Práctica 2')
print('='*65)
print()
print('Componentes implementados:')
print('  ✅ Chunking Semántico (SemanticChunker, percentile 95)')
print('  ✅ HyDE (consultas vagas < 10 palabras)')
print('  ✅ Query Decomposition (consultas multi-pregunta)')
print('  ✅ Recuperación MMR (λ=0.6, k dinámico via Groq)')
print('  ✅ Agente ReAct (LangGraph + 5 tools)')
print('  ✅ Agente Reflecting (loop ×3 + internet fallback)')
print('  ✅ Knowledge Graph RAG (ontología OWL + SPARQL + GraphDB)')
print('  ✅ LangSmith trazabilidad (nodos, tokens, latencia)')
print('  ✅ Métricas: Recall@k, Precision@k, MRR, nDCG, Relevance, Faithfulness')
print()
print('Ontología OWL (biomedical_ontology.ttl):')
print('  • 12 clases | 14 propiedades | 4+ individuos/clase')
print('  • 2 DisjointClasses | 3 owl:inverseOf')
print('  • Restricciones: allValuesFrom, someValuesFrom, minCardinality')
print('  • Constructor: owl:unionOf (BiomedicalPublication)')
print()
print('LLMs asignados:')
print('  • Gemini 2.0 Flash: ReAct, HyDE, generación, reflexión, juez')
print('  • Groq LLaMA 3.3 70B: k dinámico, decomposición rápida')
print()
print(f'LangSmith: https://smith.langchain.com → {os.environ["LANGCHAIN_PROJECT"]}')
print('GraphDB:   http://localhost:7200/repositories/biomed-kg')
